# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaimAli0001/Flyrank-Internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector

The feature vector contains one row per content page for a single client using aggregated search performance from March 2026.

The selected features were chosen during the data contract stage because they are available before the prediction point and describe the search performance of each page.

The feature vector contains:

- `march_impressions`
- `march_ctr`
- `march_avg_position`
- `impression_trend`
- `ctr_trend`

The context fields `client_hash_id` and `content_hash_id` are retained only for identification and grouping and are not predictive features.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

token = os.getenv("HF_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [3]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")


print("DuckDB secret created")

DuckDB secret created


In [4]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [6]:
schema = con.sql(f"""
DESCRIBE SELECT * FROM {REL}
""").df()


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_vector = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- Total March impressions
    SUM(gsc_impressions) AS march_impressions,

    -- March CTR
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN (SUM(gsc_clicks) * 100.0) / SUM(gsc_impressions)
        ELSE NULL
    END AS march_ctr,

    -- Average March position
    AVG(gsc_avg_position) AS march_avg_position,

    -- Impression trend
    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
            THEN gsc_impressions
            ELSE 0
        END
    ) -
    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN gsc_impressions
            ELSE 0
        END
    ) AS impression_trend,

    -- CTR trend
    (
        CASE
            WHEN SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) > 0
            THEN
                SUM(
                    CASE
                        WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                        THEN gsc_clicks
                        ELSE 0
                    END
                ) * 100.0 /
                SUM(
                    CASE
                        WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                        THEN gsc_impressions
                        ELSE 0
                    END
                )
        END
    )
    -
    (
        CASE
            WHEN SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) > 0
            THEN
                SUM(
                    CASE
                        WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                        THEN gsc_clicks
                        ELSE 0
                    END
                ) * 100.0 /
                SUM(
                    CASE
                        WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                        THEN gsc_impressions
                        ELSE 0
                    END
                )
        END
    ) AS ctr_trend

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

feature_vector.head()

,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend
0,client_73cda7b4e4f265ea,content_b2f28253ee66597e,355.0,0.0,40.450647,75.0,0.0
1,client_73cda7b4e4f265ea,content_d8d1e998c5265ffb,64.0,0.0,8.197436,0.0,0.0
2,client_73cda7b4e4f265ea,content_fc6508d7b261d9e6,187.0,0.0,24.006864,-11.0,0.0
3,client_73cda7b4e4f265ea,content_cf5124ce3435ca8f,13.0,0.0,17.666667,9.0,0.0
4,client_73cda7b4e4f265ea,content_7b862b315066f6b1,433.0,0.0,16.644566,-31.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

| Feature | Meaning | Missing Values | Available Before Prediction? |
|---------|---------|----------------|------------------------------|
| **march_impressions** | Total Google Search impressions during March 2026. | Missing values are treated as zero because no impressions indicate no observed visibility. | Yes |
| **march_ctr** | Click-through rate calculated from March clicks and impressions. | If impressions are zero, CTR is left as missing because it cannot be calculated. | Yes |
| **march_avg_position** | Average Google Search ranking position during March 2026. | Missing values occur only when Google Search Console data is unavailable. These rows are excluded during feature construction. | Yes |
| **impression_trend** | Difference between impressions in the second half of March and the first half of March. Positive values indicate increasing visibility, while negative values indicate declining visibility. | Missing values are not expected because impressions are aggregated over the feature window. | Yes |
| **ctr_trend** | Difference between CTR in the second half of March and the first half of March. Positive values indicate improving click-through performance, while negative values indicate deterioration. | If one period has no impressions, the trend remains missing because CTR cannot be calculated reliably. | Yes |

All selected features are derived only from March 2026 data and are available before the prediction point. No future information is used when constructing the feature vector.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_vector.info()

feature_vector.head()

<class 'pandas.DataFrame'>
RangeIndex: 176738 entries, 0 to 176737
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   client_hash_id      176738 non-null  str    
 1   content_hash_id     176738 non-null  str    
 2   march_impressions   176738 non-null  float64
 3   march_ctr           176738 non-null  float64
 4   march_avg_position  176738 non-null  float64
 5   impression_trend    176738 non-null  float64
 6   ctr_trend           141467 non-null  float64
dtypes: float64(5), str(2)
memory usage: 9.4 MB


,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend
0,client_73cda7b4e4f265ea,content_b2f28253ee66597e,355.0,0.0,40.450647,75.0,0.0
1,client_73cda7b4e4f265ea,content_d8d1e998c5265ffb,64.0,0.0,8.197436,0.0,0.0
2,client_73cda7b4e4f265ea,content_fc6508d7b261d9e6,187.0,0.0,24.006864,-11.0,0.0
3,client_73cda7b4e4f265ea,content_cf5124ce3435ca8f,13.0,0.0,17.666667,9.0,0.0
4,client_73cda7b4e4f265ea,content_7b862b315066f6b1,433.0,0.0,16.644566,-31.0,0.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Feature Notes

The feature vector contains five engineered features derived only from March 2026 search performance. These features are available before the prediction point and therefore do not introduce future information.

| Feature | Meaning | Missing Value Handling | Available Before Prediction? |
|---------|---------|------------------------|------------------------------|
| **march_impressions** | Total Google Search impressions during March 2026. | Missing values are treated as zero because no impressions indicate no observed visibility. | Yes |
| **march_ctr** | Click-through rate calculated using March clicks and impressions. | If total impressions are zero, CTR remains missing because it cannot be calculated. | Yes |
| **march_avg_position** | Average Google Search ranking position during March 2026. Lower values indicate better rankings. | Missing values occur only when Google Search Console data is unavailable. Those rows are excluded during feature construction. | Yes |
| **impression_trend** | Difference between impressions during the second half of March and the first half of March. Positive values indicate increasing visibility, while negative values indicate declining visibility. | Missing values are not expected because impressions are aggregated across the feature window. | Yes |
| **ctr_trend** | Difference between CTR during the second half of March and the first half of March. Positive values indicate improving click performance, while negative values indicate deterioration. | If one of the two periods has zero impressions, the trend remains missing because CTR cannot be calculated reliably. | Yes |

All selected features are computed exclusively from March 2026 data and are available before the prediction point, making them suitable for downstream modelling.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the engineered feature vector

feature_vector.info()

feature_vector.head()

<class 'pandas.DataFrame'>
RangeIndex: 176738 entries, 0 to 176737
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   client_hash_id      176738 non-null  str    
 1   content_hash_id     176738 non-null  str    
 2   march_impressions   176738 non-null  float64
 3   march_ctr           176738 non-null  float64
 4   march_avg_position  176738 non-null  float64
 5   impression_trend    176738 non-null  float64
 6   ctr_trend           141467 non-null  float64
dtypes: float64(5), str(2)
memory usage: 9.4 MB


,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend
0,client_73cda7b4e4f265ea,content_b2f28253ee66597e,355.0,0.0,40.450647,75.0,0.0
1,client_73cda7b4e4f265ea,content_d8d1e998c5265ffb,64.0,0.0,8.197436,0.0,0.0
2,client_73cda7b4e4f265ea,content_fc6508d7b261d9e6,187.0,0.0,24.006864,-11.0,0.0
3,client_73cda7b4e4f265ea,content_cf5124ce3435ca8f,13.0,0.0,17.666667,9.0,0.0
4,client_73cda7b4e4f265ea,content_7b862b315066f6b1,433.0,0.0,16.644566,-31.0,0.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields

The following fields were intentionally excluded from the feature vector.

| Excluded Field | Reason |
|---------------|--------|
| **GA4 engagement metrics** | Coverage is limited across pages, making them inconsistent for modelling. |
| **Search volume** | Represents keyword demand rather than the observed performance of the page. Search performance metrics provide stronger evidence for the refresh decision. |
| **Future months (April onwards)** | Not available at prediction time and would introduce data leakage. |
| **Optimization dates** | These fields may contain information that occurs after the feature window and could leak future knowledge into the model. |
| **Proxy label (Refresh Priority)** | Represents the target decision and must never be used as an input feature. |
| **Any label-derived or outcome fields** | These directly encode the answer the model is expected to predict, resulting in target leakage. |

These exclusions ensure that the feature vector contains only information that would have been available when making the refresh decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.